In [ ]:
import numpy as np
from aeon.transformations.collection import (
    ARCoefficientTransformer,
    PeriodogramTransformer,
)
from aeon.transformations.collection.interval_based import SupervisedIntervals
from aeon.utils.numba.general import first_order_differences_3d
from aeon.utils.validation import check_n_jobs
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

class RSTSFTransformer:
    def __init__(self, n_intervals=50, min_interval_length=3, random_state=None, n_jobs=1):
        self.n_intervals = n_intervals
        self.min_interval_length = min_interval_length
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y):
        self._n_jobs = check_n_jobs(self.n_jobs)
        lags = int(12 * (X.shape[2] / 100.0) ** 0.25)

        self._series_transformers = [
            FunctionTransformer(func=first_order_differences_3d, validate=False),
            PeriodogramTransformer(),
            ARCoefficientTransformer(order=lags, replace_nan=True),
        ]

        transforms = [X] + [t.fit_transform(X) for t in self._series_transformers]

        self._transformers = []
        for t in transforms:
            si = SupervisedIntervals(
                n_intervals=self.n_intervals,
                min_interval_length=self.min_interval_length,
                n_jobs=self._n_jobs,
                random_state=self.random_state,
                randomised_split_point=True,
            )
            si.fit(t, y)
            self._transformers.append(si)

        return self

    def transform(self, X):
        transforms = [X] + [t.transform(X) for t in self._series_transformers]

        Xt = np.empty((X.shape[0], 0))
        for i, t in enumerate(transforms):
            Xt = np.hstack((Xt, self._transformers[i].transform(t)))

        return Xt

    def fit_transform(self, X, y):
        self.fit(X, y)

        transforms = [X] + [t.transform(X) for t in self._series_transformers]

        Xt = np.empty((X.shape[0], 0))
        for i, t in enumerate(transforms):
            Xt = np.hstack((Xt, self._transformers[i].transform(t)))

        return Xt

In [2]:
from aeon.transformations.collection.interval_based import RandomIntervals


class RSTSFUnsupervisedTransformer:
    def __init__(self, n_intervals=50, random_state=None, n_jobs=1):
        self.n_intervals = n_intervals
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y=None):
        lags = int(12 * (X.shape[2] / 100.0) ** 0.25)

        self._series_transformers = [
            FunctionTransformer(func=first_order_differences_3d, validate=False),
            PeriodogramTransformer(),
            ARCoefficientTransformer(order=lags, replace_nan=True),
        ]

        transforms = [X] + [t.fit_transform(X) for t in self._series_transformers]

        self._transformers = []
        for t in transforms:
            ri = RandomIntervals(
                n_intervals=self.n_intervals,
                random_state=self.random_state,
                n_jobs=self.n_jobs,
                dilation=[1, 2, 4],
            )
            ri.fit(t)
            self._transformers.append(ri)

        return self

    def transform(self, X):
        transforms = [X] + [t.transform(X) for t in self._series_transformers]
        return np.hstack([self._transformers[i].transform(t) for i, t in enumerate(transforms)])

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)

In [3]:
from autotsc import utils

dataset = "CricketX"
dataset = "GestureMidAirD1"
# dataset = 'Symbols'
dataset = "ProximalPhalanxOutlineCorrect"
# dataset = 'ElectricDevices'

X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(600, 1, 80) (600,) (291, 1, 80) (291,)


In [4]:
from autotsc.models2 import RSTSFUnsupervisedClassifier


m = RSTSFUnsupervisedClassifier(n_intervals=200, random_state=42, n_jobs=-1)
m.fit(X_train, y_train)
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

NameError: name 'accuracy_score' is not defined

In [ ]:
t1 = RSTSFUnsupervisedTransformer(n_intervals=1500, n_jobs=-1)
X_train_t = t1.fit_transform(X_train)
X_test_t = t1.transform(X_test)
print(X_train_t.shape, X_test_t.shape)

In [ ]:
t2 = RSTSFTransformer(n_intervals=100, n_jobs=-1)
X_train_t2 = t2.fit_transform(X_train, y_train)
X_test_t2 = t2.transform(X_test)
print(X_train_t2.shape, X_test_t2.shape)

In [ ]:
m1 = ExtraTreesClassifier(n_jobs=-1, n_estimators=900)
m2 = ExtraTreesClassifier(n_jobs=-1, n_estimators=100)
m3 = Pipeline(
    [
        ("select", SelectKBest(f_classif, k=5000)),
        ("clf", ExtraTreesClassifier(n_jobs=-1, n_estimators=900)),
    ]
)
m4 = Pipeline(
    [
        ("select", SelectKBest(mutual_info_classif, k=5000)),
        ("clf", ExtraTreesClassifier(n_jobs=-1, n_estimators=900)),
    ]
)
m5 = RandomForestClassifier(n_jobs=-1, n_estimators=900)
m6 = make_pipeline(
    StandardScaler(),
    RidgeClassifierCV(),
)
m7 = ExtraTreesClassifier(
    n_estimators=200,
    criterion="entropy",
    class_weight="balanced",
    max_features="sqrt",
    n_jobs=-1,
)

In [ ]:


for m in [m1, m2, m3, m4, m5, m6, m7]:
    m.fit(X_train_t, y_train)
    pred = m.predict(X_test_t)
    acc = accuracy_score(y_test, pred)
    print(f"{m}: {acc:.4f}")

In [ ]:
from aeon.classification.interval_based import RSTSF
m = RSTSF(n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"{m}: {acc:.4f}")

In [ ]:
X_train_t.shape

In [ ]:
sdfsdf=sdfsf

In [ ]:
m1 = ExtraTreesClassifier(n_jobs=-1, n_estimators=900)
m1.fit(X_train_t, y_train)
acc1 = m1.score(X_test_t, y_test)
print(f"Accuracy on {dataset}: {acc1}")

In [ ]:
m2 = ExtraTreesClassifier(n_jobs=-1, n_estimators=200)
m2.fit(X_train_t2, y_train)
acc2 = m2.score(X_test_t2, y_test)
print(f"Accuracy on {dataset}: {acc2}")

In [ ]:


pipeline = Pipeline(
    [
        ("select", SelectKBest(f_classif, k=5000)),
        ("clf", ExtraTreesClassifier(n_jobs=-1, n_estimators=900)),
    ]
)
pipeline.fit(X_train_t, y_train)
acc3 = pipeline.score(X_test_t, y_test)
print(f"Accuracy on {dataset} with feature selection: {acc3}")

In [ ]:
pipeline = Pipeline(
    [
        ("select", SelectKBest(mutual_info_classif, k=5000)),
        ("clf", ExtraTreesClassifier(n_jobs=-1, n_estimators=900)),
    ]
)
pipeline.fit(X_train_t, y_train)
acc3 = pipeline.score(X_test_t, y_test)
print(f"Accuracy on {dataset} with feature selection: {acc3}")

In [ ]:
pipeline = Pipeline(
    [
        ("clf", ExtraTreesClassifier(n_jobs=-1, n_estimators=900)),
    ]
)
pipeline.fit(X_train_t, y_train)
acc3 = pipeline.score(X_test_t, y_test)
print(f"Accuracy on {dataset} with feature selection: {acc3}")

In [ ]:

pipeline = Pipeline(
    [
        # ('select', SelectKBest(mutual_info_classif, k=5000)),
        ("clf", RandomForestClassifier(n_jobs=-1, n_estimators=900)),
    ]
)
pipeline.fit(X_train_t, y_train)
acc3 = pipeline.score(X_test_t, y_test)
print(f"Accuracy on {dataset} with feature selection: {acc3}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline(
    [
        # ('select', SelectKBest(mutual_info_classif, k=5000)),
        ("clf", RandomForestClassifier(n_jobs=-1, n_estimators=900, ccp_alpha=0.0005)),
    ]
)
pipeline.fit(X_train_t, y_train)
acc3 = pipeline.score(X_test_t, y_test)
print(f"Accuracy on {dataset} with feature selection: {acc3}")

In [ ]:
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline = make_pipeline(
    StandardScaler(),
    RidgeClassifierCV(),
)
pipeline.fit(X_train_t, y_train)
acc4 = pipeline.score(X_test_t, y_test)
print(f"Accuracy on {dataset} with RidgeClassifierCV: {acc4}")

In [ ]:
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline(
    [
        # ('pca', PCA(n_components=0.95)),
        ("clf", RandomForestClassifier(n_jobs=-1, n_estimators=900)),
    ]
)
pipeline.fit(X_train_t, y_train)
acc4 = pipeline.score(X_test_t, y_test)
print(f"Accuracy on {dataset} with RidgeClassifierCV: {acc4}")

In [ ]:
from aeon.classification.shapelet_based import RSASTClassifier

clf = RSASTClassifier(n_jobs=-1)
clf.fit(X_train, y_train)
acc5 = clf.score(X_test, y_test)
print(f"Accuracy on {dataset} with RSASTClassifier: {acc5}")

In [ ]:
from aeon.classification.shapelet_based import RSASTClassifier

In [ ]:
from aeon.classification.dictionary_based import WEASEL_V2

clf = WEASEL_V2(n_jobs=-1)
clf.fit(X_train, y_train)
acc6 = clf.score(X_test, y_test)
print(f"Accuracy on {dataset} with WEASEL_V2: {acc6}")

In [ ]:
t1 = RSTSFUnsupervisedTransformer(n_intervals=100, n_jobs=1)
X_train_t = t1.fit_transform(X_train)
X_test_t = t1.transform(X_test)
print(X_train_t.shape, X_test_t.shape)

In [ ]:
import pickle
import time

from sklearn.ensemble import RandomForestClassifier

# Generate fake data
X = np.random.rand(10_000, 1_000).astype(np.float32)
y = np.random.randint(0, 2, 10_000)

print(f"X shape: {X.shape}")
print(f"X size in memory: {X.nbytes / 1e6:.2f} MB")

# Train random forest
start = time.perf_counter()
clf = RandomForestClassifier(n_estimators=500, n_jobs=-1, ccp_alpha=0.001)
clf.fit(X, y)
elapsed = time.perf_counter() - start
print(f"Training time: {elapsed:.2f}s")

# Check model size
model_bytes = len(pickle.dumps(clf))
print(f"Model size (pickled): {model_bytes / 1e6:.2f} MB")

# Count nodes across all trees
total_nodes = sum(tree.tree_.node_count for tree in clf.estimators_)
print(f"Total nodes across all trees: {total_nodes:,}")

In [ ]:
import pickle
import time

import numpy as np
from sklearn.ensemble import ExtraTreesClassifier

# Generate fake data
X = np.random.rand(10_000, 1_000).astype(np.float32)
y = np.random.randint(0, 2, 10_000)

print(f"X shape: {X.shape}")
print(f"X size in memory: {X.nbytes / 1e6:.2f} MB")

# Train random forest
start = time.perf_counter()
clf = ExtraTreesClassifier(n_estimators=500, n_jobs=-1, max_leaf_nodes=2000)
clf.fit(X, y)
elapsed = time.perf_counter() - start
print(f"Training time: {elapsed:.2f}s")

# Check model size
model_bytes = len(pickle.dumps(clf))
print(f"Model size (pickled): {model_bytes / 1e6:.2f} MB")

# Count nodes across all trees
total_nodes = sum(tree.tree_.node_count for tree in clf.estimators_)
print(f"Total nodes across all trees: {total_nodes:,}")